# 01 · Eksplorasi data backend

Data dibaca langsung dari database backend dengan **`pandas.read_sql`** (SQLAlchemy). Koneksi diatur di `.env` (`DB_URL` atau `DB_HOST`/`DB_USER`/...). Notebook ini tidak mengubah database sama sekali.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.io as pio

from dashboard import charts, data, metrics

pio.renderers.default = "notebook_connected"
pd.options.display.float_format = "{:,.2f}".format

## Koneksi dan ukuran tabel

In [ ]:
data.table_counts().to_frame()

## `read_sql` langsung

Contoh query manual memakai engine yang sama.

In [ ]:
query = """
SELECT d.name, a.alert_type, COUNT(*) AS jumlah
FROM devices_alertlog a JOIN devices_device d ON d.id = a.device_id
GROUP BY d.name, a.alert_type
ORDER BY jumlah DESC
"""
pd.read_sql(query, data.get_engine())

## Fungsi pemuat (dipakai juga oleh dashboard)

Waktu di database berformat UTC; pemuat mengonversinya ke Asia/Jakarta.

In [ ]:
data.data_bounds()

In [ ]:
devices = data.load_devices()
devices[['label', 'is_online', 'last_seen', 'tolerant_distance_cm', 'timeout_seconds']]

In [ ]:
data.load_units()

## Alert

In [ ]:
alerts = data.load_alerts()
alerts.info()
alerts.head()

In [ ]:
alerts[['occurrences', 'duration_min', 'time_to_resolve_min', 'distance_cm']].describe()

## Telemetri

Tabel telemetri besar (satu baris per unit tiap ~5 detik). `load_telemetry` mengambil data **terbaru** sampai `max_rows`.

In [ ]:
tel = data.load_telemetry(max_rows=300_000)
print(f'{len(tel):,} baris | terpotong: {tel.attrs["truncated"]}')
tel.head()

In [ ]:
tel.groupby('unit_label').agg(bacaan=('created_at', 'size'), jarak_median=('distance_cm', 'median'), tanpa_echo=('distance_cm', lambda s: s.isna().mean() * 100))